In [0]:
import os
import sys


SRC_PATH = os.path.abspath(
    "../src"
)

if SRC_PATH not in sys.path:
    sys.path.insert(
        0,
        SRC_PATH
    )

print(
    f"Project source path: {SRC_PATH}"
)

from sample_assignemnt.ingestion import (
    RawDataIngestion
)



Project source path: /Workspace/Users/manisha.tech.dey@outlook.com/sales_project_simplified_new/src


In [0]:
CATALOG_NAME = "retail_sales"

VOLUME_PATH = (
    "/Volumes/retail_sales/raw/"
)

print("=" * 70)
print("RETAIL SALES RAW INGESTION")
print("=" * 70)
print(f"Catalog     : {CATALOG_NAME}")
print(f"Volume path : {VOLUME_PATH}")
print("=" * 70)

RETAIL SALES RAW INGESTION
Catalog     : retail_sales
Volume path : /Volumes/retail_sales/raw/


In [0]:
ingestion_pipeline = RawDataIngestion(
    spark=spark,
    volume_path=VOLUME_PATH,
    catalog=CATALOG_NAME,
)

ingestion_result = (
    ingestion_pipeline.run()
)

Ingesting source : products
Target table     : retail_sales.raw.products
products: 1851 rows written to retail_sales.raw.products
Ingesting source : orders
Target table     : retail_sales.raw.orders
orders: 9994 rows written to retail_sales.raw.orders


In [0]:
summary_rows = [
    (
        source_name,
        source_result["table"],
        source_result["rows_written"],
        source_result["status"],
    )
    for source_name, source_result
    in ingestion_result.items()
]

summary_df = spark.createDataFrame(
    summary_rows,
    [
        "source_name",
        "target_table",
        "rows_written",
        "status",
    ],
)

display(
    summary_df
)


source_name,target_table,rows_written,status
products,retail_sales.raw.products,1851,SUCCESS
orders,retail_sales.raw.orders,9994,SUCCESS


In [0]:
failed_sources = [
    source_name
    for source_name, source_result
    in ingestion_result.items()
    if source_result["status"] != "SUCCESS"
]

if failed_sources:

    raise RuntimeError(
        "Raw ingestion failed for: "
        + ", ".join(failed_sources)
    )

print(
    "All source files were ingested successfully."
)


All source files were ingested successfully.


In [0]:
expected_raw_tables = [
    f"{CATALOG_NAME}.raw.customers",
    f"{CATALOG_NAME}.raw.products",
    f"{CATALOG_NAME}.raw.orders",
]

missing_tables = [
    table_name
    for table_name in expected_raw_tables
    if not spark.catalog.tableExists(
        table_name
    )
]

if missing_tables:

    raise RuntimeError(
        "The following Raw tables are missing: "
        + ", ".join(missing_tables)
    )

print(
    "All expected Raw tables exist."
)

---------------------------------------------------------------------------
RuntimeError                              Traceback (most recent call last)
File <command-8809466822354849>, line 17
      7 missing_tables = [
      8     table_name
      9     for table_name in expected_raw_tables
   (...)
     12     )
     13 ]
     15 if missing_tables:
---> 17     raise RuntimeError(
     18         "The following Raw tables are missing: "
     19         + ", ".join(missing_tables)
     20     )
     22 print(
     23     "All expected Raw tables exist."
     24 )

RuntimeError: The following Raw tables are missing: retail_sales.raw.customers

In [0]:
display(
    spark.sql(
        f"""
        SHOW TABLES IN {CATALOG_NAME}.raw
        """
    )
)

database,tableName,isTemporary
raw,orders,false
raw,products,false


In [0]:
print()
print("=" * 70)
print("RAW INGESTION PIPELINE COMPLETED SUCCESSFULLY")
print("=" * 70)

for source_name, source_result in (
    ingestion_result.items()
):

    print(
        f"{source_name:<12} | "
        f"{source_result['rows_written']:>6} rows | "
        f"{source_result['table']}"
    )

print("=" * 70)


RAW INGESTION PIPELINE COMPLETED SUCCESSFULLY
products     |   1851 rows | retail_sales.raw.products
orders       |   9994 rows | retail_sales.raw.orders
